In [ ]:
# -*- coding: utf-8 -*-
"""
三粒子横场伊辛模型（TFI）的 one-site DMRG 求解
基于 vMPS 框架，完整修复版本
"""

import numpy as np
from scipy.sparse.linalg import eigsh
from scipy.linalg import svd, qr

# ========================================
# 1. 泡利矩阵
# ========================================
def pauli_matrices():
    I = np.array([[1, 0], [0, 1]], dtype=complex)
    X = np.array([[0, 1], [1, 0]], dtype=complex)
    Z = np.array([[1, 0], [0, -1]], dtype=complex)
    return I, X, Z

I, X, Z = pauli_matrices()

# ========================================
# 2. 构造三粒子 TFI 哈密顿量（开边界）
# H = -J Σ σz_i σz_{i+1} - h Σ σx_i
# ========================================
def build_hamiltonian(J=1.0, h=1.0):
    N = 3
    H = np.zeros((2**N, 2**N), dtype=complex)

    # 相邻 σz_i σz_{i+1} 项 (i=0,1)
    for i in range(2):
        op = 1
        for j in range(3):
            if j == i or j == i+1:
                op = np.kron(op, Z)
            else:
                op = np.kron(op, I)
        H -= J * op

    # 横向场 σx_i 项
    for i in range(3):
        op = 1
        for j in range(3):
            if j == i:
                op = np.kron(op, X)
            else:
                op = np.kron(op, I)
        H -= h * op

    return H

# ========================================
# 3. 改进的 MPS 类
# ========================================
class MPS:
    def __init__(self, d=2, D=2, N=3):
        self.d = d  # 物理维度
        self.D = D  # 虚拟维度
        self.N = N  # 格点数

        # 初始化为全零态 |000⟩ + 小随机扰动
        self.A = []
        
        # 第一个张量 (1, d, D)
        A0 = np.zeros((1, d, D), dtype=complex)
        A0[0, 0, :] = 1.0  # |0⟩ 态
        A0 += 0.01 * (np.random.randn(1, d, D) + 1j * np.random.randn(1, d, D))
        self.A.append(A0)
        
        # 中间张量 (D, d, D)
        for i in range(1, N-1):
            Ai = np.zeros((D, d, D), dtype=complex)
            Ai[:, 0, :] = np.eye(D)  # |0⟩ 态
            Ai += 0.01 * (np.random.randn(D, d, D) + 1j * np.random.randn(D, d, D))
            self.A.append(Ai)
        
        # 最后一个张量 (D, d, 1)
        AN = np.zeros((D, d, 1), dtype=complex)
        AN[:, 0, 0] = 1.0  # |0⟩ 态
        AN += 0.01 * (np.random.randn(D, d, 1) + 1j * np.random.randn(D, d, 1))
        self.A.append(AN)
        
        self.normalize()

    def left_canonicalize(self, site):
        """左规范化指定位点"""
        if site >= self.N:
            return
            
        A = self.A[site]
        Dl, d, Dr = A.shape
        
        # 重塑为矩阵 (Dl*d, Dr)
        A_mat = A.reshape(Dl * d, Dr)
        
        # QR 分解
        Q, R = qr(A_mat)
        
        # 更新当前张量
        self.A[site] = Q.reshape(Dl, d, -1)
        
        # 将 R 传递给右边
        if site + 1 < self.N:
            self.A[site + 1] = np.einsum('ij,jkl->ikl', R, self.A[site + 1])

    def right_canonicalize(self, site):
        """右规范化指定位点"""
        if site < 0:
            return
            
        A = self.A[site]
        Dl, d, Dr = A.shape
        
        # 重塑为矩阵 (Dl, d*Dr)
        A_mat = A.reshape(Dl, d * Dr)
        
        # QR 分解（转置）
        Q, R = qr(A_mat.T)
        Q = Q.T
        R = R.T
        
        # 更新当前张量
        self.A[site] = Q.reshape(-1, d, Dr)
        
        # 将 R 传递给左边
        if site - 1 >= 0:
            self.A[site - 1] = np.einsum('ijk,kl->ijl', self.A[site - 1], R)

    def normalize(self):
        """归一化整个 MPS"""
        # 计算模
        norm = self.compute_norm()
        if norm > 1e-14:
            # 将归一化因子均匀分布到所有张量
            factor = norm ** (1.0 / self.N)
            for i in range(self.N):
                self.A[i] /= factor

    def compute_norm(self):
        """计算 MPS 的模"""
        # 从右向左收缩
        R = np.array([[1.0]], dtype=complex)
        
        for i in reversed(range(self.N)):
            A = self.A[i]  # (Dl, d, Dr)
            new_R = np.zeros((A.shape[0], A.shape[0]), dtype=complex)
            
            for s in range(self.d):
                As = A[:, s, :]  # (Dl, Dr)
                new_R += As @ R @ As.conj().T
            
            R = new_R
        
        return np.sqrt(np.real(R[0, 0]))

    def to_state_vector(self):
        """转换为态向量"""
        psi = np.zeros(2**self.N, dtype=complex)
        
        for idx in range(2**self.N):
            # 解码自旋配置
            spins = []
            temp_idx = idx
            for i in range(self.N):
                spins.append(temp_idx % 2)
                temp_idx //= 2
            spins.reverse()
            
            # 收缩张量
            temp = self.A[0][0, spins[0], :]  # (D,)
            for i in range(1, self.N-1):
                temp = np.dot(temp, self.A[i][:, spins[i], :])  # (D,)
            temp = np.dot(temp, self.A[self.N-1][:, spins[self.N-1], 0])  # scalar
            
            psi[idx] = temp
        
        return psi

# ========================================
# 4. 构造有效哈密顿量（改进版）
# ========================================
def build_effective_hamiltonian(mps, H_full, site):
    """构造有效哈密顿量"""
    d = mps.d
    Dl, Dr = mps.A[site].shape[0], mps.A[site].shape[2]
    
    # 构造左环境和右环境
    left_env = build_left_environment(mps, site)
    right_env = build_right_environment(mps, site)
    
    # 构造有效哈密顿量
    dim = Dl * d * Dr
    H_eff = np.zeros((dim, dim), dtype=complex)
    
    # 保存原始张量
    A_orig = mps.A[site].copy()
    
    # 遍历所有基态
    for i in range(dim):
        # 第i个基态
        T_i = np.zeros((Dl, d, Dr), dtype=complex)
        idx = i
        b = idx % Dr; idx //= Dr
        s = idx % d; idx //= d
        a = idx
        T_i[a, s, b] = 1.0
        mps.A[site] = T_i
        psi_i = mps.to_state_vector()
        
        for j in range(dim):
            # 第j个基态
            T_j = np.zeros((Dl, d, Dr), dtype=complex)
            idx = j
            b = idx % Dr; idx //= Dr
            s = idx % d; idx //= d
            a = idx
            T_j[a, s, b] = 1.0
            mps.A[site] = T_j
            psi_j = mps.to_state_vector()
            
            # 计算矩阵元
            H_eff[i, j] = np.vdot(psi_i, H_full @ psi_j)
    
    # 恢复原始张量
    mps.A[site] = A_orig
    
    return H_eff.real

def build_left_environment(mps, site):
    """构造左环境（占位符，简化版本）"""
    return np.eye(mps.A[site].shape[0])

def build_right_environment(mps, site):
    """构造右环境（占位符，简化版本）"""
    return np.eye(mps.A[site].shape[2])

# ========================================
# 5. DMRG 主循环（改进版）
# ========================================
def one_site_dmrg(H_full, mps, max_sweeps=50, tol=1e-10):
    """单点 DMRG 算法"""
    energies = []
    
    for sweep in range(max_sweeps):
        # 左扫：0 -> N-1
        for site in range(mps.N):
            # 左规范化到当前位点
            for i in range(site):
                mps.left_canonicalize(i)
            
            # 构造有效哈密顿量
            H_eff = build_effective_hamiltonian(mps, H_full, site)
            
            # 求解最小本征值问题
            try:
                eigenvals, eigenvecs = eigsh(H_eff, k=1, which='SA')
                energy = eigenvals[0]
                new_tensor = eigenvecs[:, 0].reshape(mps.A[site].shape)
                mps.A[site] = new_tensor
            except Exception as e:
                print(f"Warning: eigsh failed at site {site}: {e}")
                continue
        
        # 右扫：N-1 -> 0
        for site in range(mps.N-1, -1, -1):
            # 右规范化到当前位点
            for i in range(mps.N-1, site, -1):
                mps.right_canonicalize(i)
            
            # 构造有效哈密顿量
            H_eff = build_effective_hamiltonian(mps, H_full, site)
            
            # 求解最小本征值问题
            try:
                eigenvals, eigenvecs = eigsh(H_eff, k=1, which='SA')
                energy = eigenvals[0]
                new_tensor = eigenvecs[:, 0].reshape(mps.A[site].shape)
                mps.A[site] = new_tensor
            except Exception as e:
                print(f"Warning: eigsh failed at site {site}: {e}")
                continue
        
        # 归一化
        mps.normalize()
        
        # 计算当前能量
        psi_current = mps.to_state_vector()
        psi_current /= np.linalg.norm(psi_current)
        current_energy = np.real(np.vdot(psi_current, H_full @ psi_current))
        
        energies.append(current_energy)
        print(f"Sweep {sweep+1:2d}, Energy = {current_energy: .10f}")
        
        # 检查收敛
        if len(energies) > 1 and abs(energies[-1] - energies[-2]) < tol:
            print(f"✅ Converged at sweep {sweep+1}")
            break
    
    return energies

# ========================================
# 6. 主程序
# ========================================
if __name__ == "__main__":
    # 模型参数
    J = 1.0
    h = 1.0

    print("🔄 Building Hamiltonian...")
    H_full = build_hamiltonian(J, h)
    print(f"Hamiltonian shape: {H_full.shape}")

    # 精确对角化
    print("\n🔍 Diagonalizing full Hamiltonian...")
    e_exact, v_exact = np.linalg.eigh(H_full)
    print(f"Exact ground energy: {e_exact[0]: .10f}")

    # 初始化 MPS
    print("\n🧠 Initializing MPS...")
    mps = MPS(d=2, D=4, N=3)  # 增大虚拟维度

    # 验证初始化
    psi0 = mps.to_state_vector()
    print(f"Initial state vector norm: {np.linalg.norm(psi0):.2e}")
    print(f"MPS norm (contracted): {mps.compute_norm():.2e}")

    # 运行 DMRG
    print("\n🚀 Running one-site DMRG...")
    energies = one_site_dmrg(H_full, mps, max_sweeps=30, tol=1e-10)

    # 最终结果
    psi_dmrg = mps.to_state_vector()
    psi_dmrg /= np.linalg.norm(psi_dmrg)
    overlap = abs(np.vdot(psi_dmrg, v_exact[:, 0]))

    print(f"\n🎉 DMRG ground energy: {energies[-1]: .10f}")
    print(f"Exact ground energy:  {e_exact[0]: .10f}")
    print(f"Error: {abs(energies[-1] - e_exact[0]):.2e}")
    print(f"Overlap with exact GS: {overlap: .10f}")

In [ ]:
import torch as tc
import numpy as np
import math as mt
from tqdm import tqdm
import matplotlib.pyplot as plt

def choose_device():
    if tc.cuda.is_available():
        return 'cuda'
    else:
        return 'cpu'

def clone_func(tensors):
    """辅助函数：克隆 MPS 张量列表"""
    return [tensor.clone() for tensor in tensors]

class MPS:

    def __init__(self, length, dim, chi, device='cpu', dtype=tc.complex128, boundary='open'):
        super().__init__()
        self.length = length
        self.dim = dim
        self.chi = chi
        self.device = device
        self.dtype = dtype
        self.boundary = boundary

        if self.boundary == 'open':
            self.tensors = [tc.randn((1, dim, chi), device=device, dtype=dtype, requires_grad=True)] + [tc.randn((chi, dim, chi), device=device, dtype=dtype, requires_grad=True) for _ in range(length-2)] + [tc.randn((chi, dim, 1), device=device, dtype=dtype, requires_grad=True)]
        elif self.boundary == 'periodic':
            self.tensors = [tc.randn((chi, dim, chi), device=device, dtype=dtype, requires_grad=True) for _ in range(length)]

    def clone_tensors(self):
        return clone_func(self.tensors)

    def clone_mps(self):
        psi = MPS(self.length, self.dim, self.chi, self.device, self.dtype)
        psi.tensors = clone_func(self.tensors)
        return psi

    def normalize(self):
        mps0 = self.clone_tensors()
        #v = tc.eye(mps0[0].shape[0]**2, dtype=tc.float64, device=paras['device']).reshape([mps0[0].shape[0]] * 4)
        v = tc.ones((mps0[0].shape[0], mps0[0].shape[0]), dtype=self.dtype, device=self.device)
        mps_n = []
        norm = tc.zeros(len(self.tensors), dtype=tc.float64, device=self.device)
        for i in range(len(self.tensors)):
            #v = tc.einsum('uvab,acd,bce->uvde', v, mps0[i].conj(), mps0[i])
            v = tc.einsum('ab,acd,bce->de', v, mps0[i].conj(), mps0[i])
            #n = tc.einsum('abcd->', v)
            n = tc.norm(v) + 1e-12
            mps_n.append(self.tensors[i] / tc.sqrt(n))
            v = v / n
            norm[i] = n
        norms = tc.prod(norm)
        
        # 保持计算图：原地更新每个张量而不是重新赋值列表
        for i in range(len(self.tensors)):
            self.tensors[i].data = mps_n[i].data  # 只更新数据，保持计算图
        
        return norms

    def contract_mps(self):
        """收缩整个 MPS 为单个张量"""
        tensor = self.tensors[0]
        for A in self.tensors[1:]:
            #print("收缩前tensor：",tensor.shape)
            #print("A：",A.shape)
            tensor = tc.einsum('...i,ijk->...jk', tensor, A)
            #print("收缩后tensor：",tensor.shape)
        if self.boundary == 'open':
            return tensor.squeeze()
        elif self.boundary == 'periodic':
            return tc.einsum('i...i->...', tensor)


    def add_single_opr(self, op, site):
        """在指定位置添加单体操作"""
        return tc.einsum('abc, bd -> adc', self.tensors[site], op)


    def inner_product(self, others, form='log', boundary='open'):
        tensors0 = self.tensors
        tensors1 = others.tensors
        assert tensors0[0].shape[0] == tensors0[-1].shape[-1]
        assert tensors1[0].shape[0] == tensors1[-1].shape[-1]
        assert len(tensors0) == len(tensors1)

        if boundary == 'open':
            v = tc.ones((1, 1), dtype=self.dtype, device=self.device)
            norm_tensor = tc.ones(len(tensors0), dtype=self.dtype, device=self.device)
            
            for i in range(len(tensors0)):
                v = tc.einsum('ab,acd,bce->de', v, tensors0[i].conj(), tensors1[i])
                
                # 添加安全保护
                n = tc.norm(v) + 1e-16  # 防止除以零
                if n.item() < 1e-12:
                    return tc.tensor(0.0, dtype=self.dtype, device=self.device)
                    
                v = v / n
                norm_tensor[i] = n
            
            norms = tc.prod(norm_tensor)
            return norms * v.squeeze()

        elif boundary == 'periodic':
            v0 = tc.eye(tensors0[0].shape[0], dtype=tensors0[0].dtype, device=tensors0[0].device)
            v1 = tc.eye(tensors1[0].shape[0], dtype=tensors0[0].dtype, device=tensors0[0].device)
            v = tc.kron(v0, v1).reshape([tensors0[0].shape[0], tensors1[0].shape[0],
                                        tensors0[0].shape[0], tensors1[0].shape[0]])
            norm_list = list()
            for n in range(len(tensors0)):
                v = tc.einsum('uvap,adb,pdq->uvbq', v, tensors0[n].conj(), tensors1[n])
                norm_list.append(v.norm() + 1e-10)
                v = v / norm_list[-1]
            if v.numel() > 1:
                norm1 = tc.einsum('acac->', v)
                norm_list.append(norm1 + 1e-10)
            else:
                norm_list.append(v[0, 0, 0, 0] + 1e-10)
            if form == 'log':  # 返回模方的log，舍弃符号
                norm = 0.0
                for x in norm_list:
                    norm = norm + tc.log(x.abs())
            elif form == 'list':  # 返回列表
                return norm_list
            else:  # 直接返回模方
                norm = 1.0
                for x in norm_list:
                    norm = norm * x
            return norm

    def energy_calcu(self, ops, sites):
        """计算给定操作和位置的能量期望值"""
        assert len(ops) == len(sites), "操作和位置列表长度不匹配"
        energy = 0
        psi = self.clone_mps()
        for op, site in zip(ops, sites):
            psi.tensors[site] = psi.add_single_opr(op, site)
        energy = self.inner_product(psi, boundary=self.boundary)
        return energy
    
    def energy_with_MPO(self, mpo):
        assert self.length == mpo.length, "MPS和MPO长度不匹配"
    
        if self.boundary == 'open':
             v = tc.ones((1, 1, 1), dtype=self.dtype, device=self.device)
             norm_tensor = tc.ones(self.length, dtype=self.dtype, device=self.device)
            
             for i in range(self.length):
                 #print(f"shape of self.tensors[{i}]:", self.tensors[i].shape)
                 #print(f"shape of mpo.tensors[{i}]:", mpo.tensors[i].shape)
                 v = tc.einsum('abc, adi, cek, bjde -> ijk', v, self.tensors[i].conj(), self.tensors[i], mpo.tensors[i])
                 # 添加安全保护
                 n = tc.norm(v) + 1e-16  # 防止除以零
                 if n.item() < 1e-12:
                    return tc.tensor(0.0, dtype=self.dtype, device=self.device)
                 v = v / n
                 norm_tensor[i] = n
            
             norms = tc.prod(norm_tensor)
             return norms * v.squeeze()
        
        if self.boundary == 'periodic':
            assert self.length == mpo.length

            v0 = tc.eye(self.tensors[0].shape[0], dtype=self.dtype, device=self.device)
            v1 = tc.eye(self.tensors[0].shape[0], dtype=self.dtype, device=self.device)
            v = tc.kron(v0, v1).reshape([self.tensors[0].shape[0], self.tensors[0].shape[0],
                                        self.tensors[0].shape[0], self.tensors[0].shape[0]])
            norm_list = list()
            for n in range(self.length):
                v = tc.einsum('uvap,adb,pdq->uvbq', v, self.tensors[n].conj(), self.tensors[n])
                norm_list.append(v.norm() + 1e-10)
                v = v / norm_list[-1]
            if v.numel() > 1:
                norm1 = tc.einsum('acac->', v)
                norm_list.append(norm1 + 1e-10)
            else:
                norm_list.append(v[0, 0, 0, 0] + 1e-10)
            if form == 'log':  # 返回模方的log，舍弃符号
                norm = 0.0
                for x in norm_list:
                    norm = norm + tc.log(x.abs())
            elif form == 'list':  # 返回列表
                return norm_list
            else:  # 直接返回模方
                norm = 1.0
                for x in norm_list:
                    norm = norm * x
            return norm
        
    def energy_with_MPO_optimized(self, mpo):
        assert self.length == mpo.length, "MPS和MPO长度不匹配"
        
        # 参数配置
        block_size = 5  # 块大小，减少迭代次数200次->40次
        virtual_dim = mpo.tensors[0].shape[0]  # MPO虚拟维度

        # 块处理优化 - 预处理张量块
        block_starts = range(0, self.length, block_size)
        mps_blocks = []
        mpo_blocks = []
        
        for start in block_starts:
            end = min(start + block_size, self.length)
            # 合并MPS块 (chi, dim^block_size, chi)
            mps_block = self.tensors[start]
            s0 = mps_block.shape[0]
            s1 = mps_block.shape[2]
            for i in range(start+1, end):
                mps_block = tc.einsum('xdp,piq->xdiq', mps_block, self.tensors[i])
                mps_block = mps_block.reshape(s0, -1, s1)
            
            # 合并MPO块 (D, D, dim^block_size, dim^block_size)
            mpo_block = mpo.tensors[start]
            for i in range(start+1, end):
                mpo_block = tc.einsum('abcd, bgef', mpo_block, mpo.tensors[i])
                k = mpo_block.shape[2] * mpo_block.shape[3] * mpo_block.shape[4] * mpo_block.shape[5]
                mpo_block = mpo_block.reshape(virtual_dim, virtual_dim, k//2, k//2)

            mps_blocks.append(mps_block)
            mpo_blocks.append(mpo_block)
        
        # 块收缩优化
        v = tc.ones((1, 1, 1), dtype=self.dtype, device=self.device)
        norm_cum = tc.ones(1, dtype=self.dtype, device=self.device)
        
        with tc.no_grad():  # 禁止梯度计算加速
            for i in range(len(mps_blocks)):
                # 批量einsum处理
                v_next = tc.einsum('abc, adi, cek, bjde -> ijk', 
                                v, 
                                mps_blocks[i].conj(),
                                mps_blocks[i],
                                mpo_blocks[i])
                
                # 稳定性处理
                norm_step = tc.norm(v_next) + 1e-16
                v = v_next / norm_step
                norm_cum *= norm_step
        
        return norm_cum * v.squeeze()
            


class MPO:
    def __init__(self, length, dim, device='cpu', dtype=tc.complex128, name=None):
        self.length = length
        self.dim = dim
        self.device = device
        self.dtype = dtype
        self.name = name
        self.tensors = None
        
    def generate_spin_operators(self):
        """
        生成自旋维度为dim的所有自旋算符矩阵
        
        Returns:
            dict: 包含所有自旋算符的字典
        """
        # 计算自旋量子数 S
        S = (self.dim - 1) / 2.0
        
        # 生成 m 量子数：从 -S 到 +S
        m_values = np.arange(S, -S-1, -1)

        # 初始化算符矩阵
        Sz = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        S_plus = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        S_minus = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        
        # Sz 算符：对角矩阵，对角元为 m 值
        for i, m in enumerate(m_values):
            Sz[i, i] = m
        
        # S+ 和 S- 算符
        for i, m in enumerate(m_values):
            if i > 0:  # S+ 连接 |m⟩ 和 |m+1⟩
                # S+|m⟩ = sqrt(S(S+1) - m(m+1))|m+1⟩
                coeff = np.sqrt(S * (S + 1) - m * (m + 1))
                S_plus[i-1, i] = coeff  # (m+1, m) 元素

            if i < self.dim - 1:  # S- 连接 |m⟩ 和 |m-1⟩
                # S-|m⟩ = sqrt(S(S+1) - m(m-1))|m-1⟩
                coeff = np.sqrt(S * (S + 1) - m * (m - 1))
                S_minus[i+1, i] = coeff  # (m-1, m) 元素
        
        # Sx 和 Sy 算符
        Sx = (S_plus + S_minus) / 2.0
        Sy = (S_plus - S_minus) / (2.0j)
        
        # 单位矩阵
        I = tc.eye(self.dim, dtype=self.dtype, device=self.device)
        
        return {
            'I': I,
            'Sx': Sx,
            'Sy': Sy, 
            'Sz': Sz,
            'S_plus': S_plus,
            'S_minus': S_minus,
            'S': S,  # 自旋量子数
            'dim': self.dim
        }
        
    def get_spin_operator(self, op_name):
        """
        获取指定的自旋算符
        
        Args:
            op_name: 算符名称 ('Sx', 'Sy', 'Sz', 'S_plus', 'S_minus', 'I')
        """
        operators = self.generate_spin_operators()
        
        if op_name in operators:
            self.tensor = operators[op_name]
            self.name = op_name
            return self.tensor
        else:
            raise ValueError(f"未知的算符名称: {op_name}")
    
    def get_pauli_matrices(self):
        """获取泡利矩阵 (自旋-1/2)"""
        if self.dim != 2:
            raise ValueError("泡利矩阵只适用于自旋-1/2系统 (dim=2)")
        
        operators = self.generate_spin_operators()
        
        # 泡利矩阵是自旋算符的2倍
        pauli_x = 2 * operators['Sx']
        pauli_y = 2 * operators['Sy'] 
        pauli_z = 2 * operators['Sz']
        
        return {
            'I': operators['I'],
            'sigma_x': pauli_x,
            'sigma_y': pauli_y,
            'sigma_z': pauli_z,
            'sigma_plus': pauli_x + 1j * pauli_y,
            'sigma_minus': pauli_x - 1j * pauli_y
        }
    
    def generate_TFI_MPO(self, J, h):
        """生成横场Ising模型的MPO张量列表"""
        #初始化一些空列表
        self.tensors = [None] * self.length

        # 物理维度
        d = self.dim
        
        # 虚拟维度：开边界时为2
        D = 3
        
        for i in range(self.length):
            # 创建MPO张量 (左虚拟, 右虚拟, 物理入, 物理出)
            mpo_tensor = tc.zeros((D, D, d, d), dtype=self.dtype, device=self.device)
            
            # 单位算符部分
            mpo_tensor[0, 0, :, :] = self.get_spin_operator('I')
            mpo_tensor[2:, 2:, :, :] = self.get_spin_operator('I')

            #Sz算符
            mpo_tensor[0, 1, :, :] = mt.sqrt(J) * self.get_spin_operator('Sz')
            mpo_tensor[1, 2, :, :] = - mt.sqrt(J) * self.get_spin_operator('Sz')

            #Sx算符
            mpo_tensor[0, 2, :, :] = h * self.get_spin_operator('Sx')
            
            # 边界条件处理
            if i == 0:  # 左边界
                mpo_tensor = mpo_tensor[0:1, :, :, :]  # 只保留第一行
            if i == self.length - 1:  # 右边界
                mpo_tensor = mpo_tensor[:, 2:3, :, :]  # 只保留第二列
            
            self.tensors[i] = mpo_tensor
            
        return self



def calc_TFI_3(mps, opr, J, h):
    "计算三粒子横场Ising model的能量期望值"
    assert mps.length == 3, "MPS长度必须为3"
    assert opr.dim == 2, "操作维度必须为2"

    # 获取Sz算符
    sz_op = opr.get_spin_operator('Sz')
    sx_op = opr.get_spin_operator('Sx')
    id_op = opr.get_spin_operator('I')

    # 直接写出mpo列表：
    mpo1 = [sz_op, sz_op, id_op]
    mpo2 = [id_op, sz_op, sz_op]
    mpo3 = [sx_op, id_op, id_op]
    mpo4 = [id_op, sx_op, id_op]
    mpo5 = [id_op, id_op, sx_op]

    mpo_list1 = [mpo1, mpo2]
    mpo_list2 = [mpo3, mpo4, mpo5]

    #计算对应的能量：
    energy = 0
    for mpo in mpo_list1:
        energy += -J * mps.energy_calcu(mpo, [0, 1, 2])
    for mpo in mpo_list2:
        energy += h * mps.energy_calcu(mpo, [0, 1, 2])

    return energy

def main_calcu():
    #输出当前使用的设备，cuda可用则输出当前gpu，否则是当前的cpu
    print(f"Using device: {choose_device()}")
    mps_i = MPS(length=3, dim=2, chi=6, device=choose_device(), dtype=tc.float64)
    spinopr = MPO(length=3, dim=2, device=choose_device(), dtype=tc.float64)

    #输出mps_i和spinor的device
    print(f"mps_i device: {mps_i.device}")
    print(f"spinopr device: {spinopr.device}")

    opti = tc.optim.Adam(mps_i.tensors, lr = 1e-1)

    step_times = 20000

    pbar = tqdm(range(step_times), desc="Variational DMRG", unit="step")

    for step in pbar:
        norm = mps_i.normalize()
        # 计算能量期望值
        energy = calc_TFI_3(mps_i, spinopr, J=1.0, h=0.5)

        # 反向传播和优化
        opti.zero_grad()
        energy.backward()
        opti.step()

        pbar.set_postfix({'Energy': f'{energy.item()}'})

def test_if_norm():
    tmps = MPS(length=3, dim=2, chi=30, device=choose_device(), dtype=tc.float64, boundary='open')
    #print(tmps.tensors)

    norm = tmps.normalize()
    print("Normalization factor:", norm.item())

    norm2 = tmps.normalize()
    print("Normalization factor (second call):", norm2.item())

    psi = tmps.clone_mps()

    factor = tmps.inner_product(psi, boundary='open')
    print("Inner product factor:", factor)

def check_calcu():
    mps_i = MPS(length=3, dim=2, chi=10, device=choose_device(),dtype=tc.float64)
    spinopr = MPO(length=3, dim=2, device=choose_device(),dtype=tc.float64)
    
    with tc.no_grad():
        # 设置全向上态 |↑↑↑⟩
        # 第一个张量 [1, 2, 10]
        mps_i.tensors[0].data = tc.zeros((1, 2, 10), device=choose_device(),dtype=tc.float64)
        mps_i.tensors[0].data[0, 0, 0] = 1.0  # |↑⟩态
        
        # 中间张量 [10, 2, 10]
        mps_i.tensors[1].data = tc.zeros((10, 2, 10), device=choose_device(),dtype=tc.float64)
        mps_i.tensors[1].data[0, 0, 0] = 1.0  # |↑⟩态
        
        # 最后一个张量 [10, 2, 1]
        mps_i.tensors[2].data = tc.zeros((10, 2, 1), device=choose_device(),dtype=tc.float64)
        mps_i.tensors[2].data[0, 0, 0] = 1.0  # |↑⟩态
    
    # 计算内积确保归一化
    norm_factor = mps_i.inner_product(mps_i.clone_mps(), boundary='open')
    print(f"初始态内积: {norm_factor.item()}")

    tensor = mps_i.contract_mps()
    print("收缩后的张量形状:", tensor.shape)
    print(tensor.reshape(8,1,1).squeeze())
    
    # 计算能量
    energy = calc_TFI_3(mps_i, spinopr, J=1.0, h=0.5)
    print("|↑↑↑⟩ Energy:", energy.item())

def calc_TFI_with_MPO():
    mps_i = MPS(length=20, dim=2, chi=50, device=choose_device(),dtype=tc.float64)
    tfimpo = MPO(length=20, dim=2, device=choose_device(),dtype=tc.float64).generate_TFI_MPO(J=1.0, h=0.5)

    opti = tc.optim.Adam(mps_i.tensors, lr=1e-4)

    steps = 10000
    t_steps = [x for x in range(1, steps + 1)]
    loss = []

    pbar = tqdm(range(steps), desc="Variational DMRG", unit="step")

    for step in pbar:
        norm = mps_i.normalize()
        energy = mps_i.energy_with_MPO(tfimpo)

        opti.zero_grad()
        energy.backward()
        opti.step()

        loss.append(energy.item())
        pbar.set_postfix({'Energy': f'{energy.item()}'})

    plt.figure(figsize=(10, 5))
    plt.plot(t_steps, loss, label='Energy variation curve')
    plt.xlabel('Steps')
    plt.ylabel('Ground state Energy')
    plt.title('Energy Variation Curve')
    plt.legend()
    plt.grid()
    plt.show()
    

if __name__ == "__main__":
    #main_calcu()
    #check_calcu()
    #test_if_norm()
    #tfi_model = MPO(length=3, dim=2, device=choose_device(), dtype=tc.float64)
    #tfi_model = tfi_model.generate_TFI_MPO(J=1.0, h=0.5)
    #print(tfi_model.tensors)
    calc_TFI_with_MPO()

In [ ]:
import torch as tc
import numpy as np
import math as mt
from tqdm import tqdm
import matplotlib.pyplot as plt

def choose_device():
    if tc.cuda.is_available():
        return 'cuda'
    else:
        return 'cpu'

def clone_func(tensors):
    """辅助函数：克隆 MPS 张量列表"""
    return [tensor.clone() for tensor in tensors]

class MPS:

    def __init__(self, length, dim, chi, device='cpu', dtype=tc.complex128, boundary='open'):
        super().__init__()
        self.length = length
        self.dim = dim
        self.chi = chi
        self.device = device
        self.dtype = dtype
        self.boundary = boundary

        if self.boundary == 'open':
            self.tensors = [tc.randn((1, dim, chi), device=device, dtype=dtype, requires_grad=True)] + [tc.randn((chi, dim, chi), device=device, dtype=dtype, requires_grad=True) for _ in range(length-2)] + [tc.randn((chi, dim, 1), device=device, dtype=dtype, requires_grad=True)]
        elif self.boundary == 'periodic':
            self.tensors = [tc.randn((chi, dim, chi), device=device, dtype=dtype, requires_grad=True) for _ in range(length)]

    def clone_tensors(self):
        return clone_func(self.tensors)

    def clone_mps(self):
        psi = MPS(self.length, self.dim, self.chi, self.device, self.dtype)
        psi.tensors = clone_func(self.tensors)
        return psi

    def normalize(self):
        mps0 = self.clone_tensors()
        #v = tc.eye(mps0[0].shape[0]**2, dtype=tc.float64, device=paras['device']).reshape([mps0[0].shape[0]] * 4)
        v = tc.ones((mps0[0].shape[0], mps0[0].shape[0]), dtype=self.dtype, device=self.device)
        mps_n = []
        norm = tc.zeros(len(self.tensors), dtype=tc.float64, device=self.device)
        for i in range(len(self.tensors)):
            #v = tc.einsum('uvab,acd,bce->uvde', v, mps0[i].conj(), mps0[i])
            v = tc.einsum('ab,acd,bce->de', v, mps0[i].conj(), mps0[i])
            #n = tc.einsum('abcd->', v)
            n = tc.norm(v) + 1e-12
            mps_n.append(self.tensors[i] / tc.sqrt(n))
            v = v / n
            norm[i] = n
        norms = tc.prod(norm)
        
        # 保持计算图：原地更新每个张量而不是重新赋值列表
        for i in range(len(self.tensors)):
            self.tensors[i].data = mps_n[i].data  # 只更新数据，保持计算图
        
        return norms

    def contract_mps(self):
        """收缩整个 MPS 为单个张量"""
        tensor = self.tensors[0]
        for A in self.tensors[1:]:
            #print("收缩前tensor：",tensor.shape)
            #print("A：",A.shape)
            tensor = tc.einsum('...i,ijk->...jk', tensor, A)
            #print("收缩后tensor：",tensor.shape)
        if self.boundary == 'open':
            return tensor.squeeze()
        elif self.boundary == 'periodic':
            return tc.einsum('i...i->...', tensor)


    def add_single_opr(self, op, site):
        """在指定位置添加单体操作"""
        return tc.einsum('abc, bd -> adc', self.tensors[site], op)


    def inner_product(self, others, form='log', boundary='open'):
        tensors0 = self.tensors
        tensors1 = others.tensors
        assert tensors0[0].shape[0] == tensors0[-1].shape[-1]
        assert tensors1[0].shape[0] == tensors1[-1].shape[-1]
        assert len(tensors0) == len(tensors1)

        if boundary == 'open':
            v = tc.ones((1, 1), dtype=self.dtype, device=self.device)
            norm_tensor = tc.ones(len(tensors0), dtype=self.dtype, device=self.device)
            
            for i in range(len(tensors0)):
                v = tc.einsum('ab,acd,bce->de', v, tensors0[i].conj(), tensors1[i])
                
                # 添加安全保护
                n = tc.norm(v) + 1e-16  # 防止除以零
                if n.item() < 1e-12:
                    return tc.tensor(0.0, dtype=self.dtype, device=self.device)
                    
                v = v / n
                norm_tensor[i] = n
            
            norms = tc.prod(norm_tensor)
            return norms * v.squeeze()

        elif boundary == 'periodic':
            v0 = tc.eye(tensors0[0].shape[0], dtype=tensors0[0].dtype, device=tensors0[0].device)
            v1 = tc.eye(tensors1[0].shape[0], dtype=tensors0[0].dtype, device=tensors0[0].device)
            v = tc.kron(v0, v1).reshape([tensors0[0].shape[0], tensors1[0].shape[0],
                                        tensors0[0].shape[0], tensors1[0].shape[0]])
            norm_list = list()
            for n in range(len(tensors0)):
                v = tc.einsum('uvap,adb,pdq->uvbq', v, tensors0[n].conj(), tensors1[n])
                norm_list.append(v.norm() + 1e-10)
                v = v / norm_list[-1]
            if v.numel() > 1:
                norm1 = tc.einsum('acac->', v)
                norm_list.append(norm1 + 1e-10)
            else:
                norm_list.append(v[0, 0, 0, 0] + 1e-10)
            if form == 'log':  # 返回模方的log，舍弃符号
                norm = 0.0
                for x in norm_list:
                    norm = norm + tc.log(x.abs())
            elif form == 'list':  # 返回列表
                return norm_list
            else:  # 直接返回模方
                norm = 1.0
                for x in norm_list:
                    norm = norm * x
            return norm

    def energy_calcu(self, ops, sites):
        """计算给定操作和位置的能量期望值"""
        assert len(ops) == len(sites), "操作和位置列表长度不匹配"
        energy = 0
        psi = self.clone_mps()
        for op, site in zip(ops, sites):
            psi.tensors[site] = psi.add_single_opr(op, site)
        energy = self.inner_product(psi, boundary=self.boundary)
        return energy
    
    def energy_with_MPO(self, mpo):
        assert self.length == mpo.length, "MPS和MPO长度不匹配"
    
        if self.boundary == 'open':
             v = tc.ones((1, 1, 1), dtype=self.dtype, device=self.device)
             norm_tensor = tc.ones(self.length, dtype=self.dtype, device=self.device)
            
             for i in range(self.length):
                 #print(f"shape of self.tensors[{i}]:", self.tensors[i].shape)
                 #print(f"shape of mpo.tensors[{i}]:", mpo.tensors[i].shape)
                 v = tc.einsum('abc, adi, cek, bjde -> ijk', v, self.tensors[i].conj(), self.tensors[i], mpo.tensors[i])
                 # 添加安全保护
                 n = tc.norm(v) + 1e-16  # 防止除以零
                 if n.item() < 1e-12:
                    return tc.tensor(0.0, dtype=self.dtype, device=self.device)
                 v = v / n
                 norm_tensor[i] = n
            
             norms = tc.prod(norm_tensor)
             return norms * v.squeeze()
        
        if self.boundary == 'periodic':
            assert self.length == mpo.length

            v0 = tc.eye(self.tensors[0].shape[0], dtype=self.dtype, device=self.device)
            v1 = tc.eye(self.tensors[0].shape[0], dtype=self.dtype, device=self.device)
            v = tc.kron(v0, v1).reshape([self.tensors[0].shape[0], self.tensors[0].shape[0],
                                        self.tensors[0].shape[0], self.tensors[0].shape[0]])
            norm_list = list()
            for n in range(self.length):
                v = tc.einsum('uvap,adb,pdq->uvbq', v, self.tensors[n].conj(), self.tensors[n])
                norm_list.append(v.norm() + 1e-10)
                v = v / norm_list[-1]
            if v.numel() > 1:
                norm1 = tc.einsum('acac->', v)
                norm_list.append(norm1 + 1e-10)
            else:
                norm_list.append(v[0, 0, 0, 0] + 1e-10)
            if form == 'log':  # 返回模方的log，舍弃符号
                norm = 0.0
                for x in norm_list:
                    norm = norm + tc.log(x.abs())
            elif form == 'list':  # 返回列表
                return norm_list
            else:  # 直接返回模方
                norm = 1.0
                for x in norm_list:
                    norm = norm * x
            return norm
        
    def energy_with_MPO_optimized(self, mpo):
        assert self.length == mpo.length, "MPS和MPO长度不匹配"
        
        # 参数配置
        block_size = 5  # 块大小，减少迭代次数200次->40次
        virtual_dim = mpo.tensors[0].shape[0]  # MPO虚拟维度

        # 块处理优化 - 预处理张量块
        block_starts = range(0, self.length, block_size)
        mps_blocks = []
        mpo_blocks = []
        
        for start in block_starts:
            end = min(start + block_size, self.length)
            # 合并MPS块 (chi, dim^block_size, chi)
            mps_block = self.tensors[start]
            s0 = mps_block.shape[0]
            s1 = mps_block.shape[2]
            for i in range(start+1, end):
                mps_block = tc.einsum('xdp,piq->xdiq', mps_block, self.tensors[i])
                mps_block = mps_block.reshape(s0, -1, s1)
            
            # 合并MPO块 (D, D, dim^block_size, dim^block_size)
            mpo_block = mpo.tensors[start]
            for i in range(start+1, end):
                mpo_block = tc.einsum('abcd, bgef', mpo_block, mpo.tensors[i])
                k = mpo_block.shape[2] * mpo_block.shape[3] * mpo_block.shape[4] * mpo_block.shape[5]
                mpo_block = mpo_block.reshape(virtual_dim, virtual_dim, k//2, k//2)

            mps_blocks.append(mps_block)
            mpo_blocks.append(mpo_block)
        
        # 块收缩优化
        v = tc.ones((1, 1, 1), dtype=self.dtype, device=self.device)
        norm_cum = tc.ones(1, dtype=self.dtype, device=self.device)
        
        with tc.no_grad():  # 禁止梯度计算加速
            for i in range(len(mps_blocks)):
                # 批量einsum处理
                v_next = tc.einsum('abc, adi, cek, bjde -> ijk', 
                                v, 
                                mps_blocks[i].conj(),
                                mps_blocks[i],
                                mpo_blocks[i])
                
                # 稳定性处理
                norm_step = tc.norm(v_next) + 1e-16
                v = v_next / norm_step
                norm_cum *= norm_step
        
        return norm_cum * v.squeeze()
            


class MPO:
    def __init__(self, length, dim, device='cpu', dtype=tc.complex128, name=None):
        self.length = length
        self.dim = dim
        self.device = device
        self.dtype = dtype
        self.name = name
        self.tensors = None
        
    def generate_spin_operators(self):
        """
        生成自旋维度为dim的所有自旋算符矩阵
        
        Returns:
            dict: 包含所有自旋算符的字典
        """
        # 计算自旋量子数 S
        S = (self.dim - 1) / 2.0
        
        # 生成 m 量子数：从 -S 到 +S
        m_values = np.arange(S, -S-1, -1)

        # 初始化算符矩阵
        Sz = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        S_plus = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        S_minus = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        
        # Sz 算符：对角矩阵，对角元为 m 值
        for i, m in enumerate(m_values):
            Sz[i, i] = m
        
        # S+ 和 S- 算符
        for i, m in enumerate(m_values):
            if i > 0:  # S+ 连接 |m⟩ 和 |m+1⟩
                # S+|m⟩ = sqrt(S(S+1) - m(m+1))|m+1⟩
                coeff = np.sqrt(S * (S + 1) - m * (m + 1))
                S_plus[i-1, i] = coeff  # (m+1, m) 元素

            if i < self.dim - 1:  # S- 连接 |m⟩ 和 |m-1⟩
                # S-|m⟩ = sqrt(S(S+1) - m(m-1))|m-1⟩
                coeff = np.sqrt(S * (S + 1) - m * (m - 1))
                S_minus[i+1, i] = coeff  # (m-1, m) 元素
        
        # Sx 和 Sy 算符
        Sx = (S_plus + S_minus) / 2.0
        Sy = (S_plus - S_minus) / (2.0j)
        
        # 单位矩阵
        I = tc.eye(self.dim, dtype=self.dtype, device=self.device)
        
        return {
            'I': I,
            'Sx': Sx,
            'Sy': Sy, 
            'Sz': Sz,
            'S_plus': S_plus,
            'S_minus': S_minus,
            'S': S,  # 自旋量子数
            'dim': self.dim
        }
        
    def get_spin_operator(self, op_name):
        """
        获取指定的自旋算符
        
        Args:
            op_name: 算符名称 ('Sx', 'Sy', 'Sz', 'S_plus', 'S_minus', 'I')
        """
        operators = self.generate_spin_operators()
        
        if op_name in operators:
            self.tensor = operators[op_name]
            self.name = op_name
            return self.tensor
        else:
            raise ValueError(f"未知的算符名称: {op_name}")
    
    def get_pauli_matrices(self):
        """获取泡利矩阵 (自旋-1/2)"""
        if self.dim != 2:
            raise ValueError("泡利矩阵只适用于自旋-1/2系统 (dim=2)")
        
        operators = self.generate_spin_operators()
        
        # 泡利矩阵是自旋算符的2倍
        pauli_x = 2 * operators['Sx']
        pauli_y = 2 * operators['Sy'] 
        pauli_z = 2 * operators['Sz']
        
        return {
            'I': operators['I'],
            'sigma_x': pauli_x,
            'sigma_y': pauli_y,
            'sigma_z': pauli_z,
            'sigma_plus': pauli_x + 1j * pauli_y,
            'sigma_minus': pauli_x - 1j * pauli_y
        }
    
    def generate_TFI_MPO(self, J, h):
        """生成横场Ising模型的MPO张量列表"""
        #初始化一些空列表
        self.tensors = [None] * self.length

        # 物理维度
        d = self.dim
        
        # 虚拟维度：开边界时为2
        D = 3
        
        for i in range(self.length):
            # 创建MPO张量 (左虚拟, 右虚拟, 物理入, 物理出)
            mpo_tensor = tc.zeros((D, D, d, d), dtype=self.dtype, device=self.device)
            
            # 单位算符部分
            mpo_tensor[0, 0, :, :] = self.get_spin_operator('I')
            mpo_tensor[2:, 2:, :, :] = self.get_spin_operator('I')

            #Sz算符
            mpo_tensor[0, 1, :, :] = mt.sqrt(J) * self.get_spin_operator('Sz')
            mpo_tensor[1, 2, :, :] = - mt.sqrt(J) * self.get_spin_operator('Sz')

            #Sx算符
            mpo_tensor[0, 2, :, :] = h * self.get_spin_operator('Sx')
            
            # 边界条件处理
            if i == 0:  # 左边界
                mpo_tensor = mpo_tensor[0:1, :, :, :]  # 只保留第一行
            if i == self.length - 1:  # 右边界
                mpo_tensor = mpo_tensor[:, 2:3, :, :]  # 只保留第二列
            
            self.tensors[i] = mpo_tensor
            
        return self

def calc_TFI_with_MPO():
    mps_i = MPS(length=20, dim=2, chi=50, device=choose_device(),dtype=tc.float64)
    tfimpo = MPO(length=20, dim=2, device=choose_device(),dtype=tc.float64).generate_TFI_MPO(J=1.0, h=0.5)

    opti = tc.optim.Adam(mps_i.tensors, lr=1e-4)

    steps = 10000
    t_steps = [x for x in range(1, steps + 1)]
    loss = []

    pbar = tqdm(range(steps), desc="Variational DMRG", unit="step")

    for step in pbar:
        norm = mps_i.normalize()
        energy = mps_i.energy_with_MPO(tfimpo)

        opti.zero_grad()
        energy.backward()
        opti.step()

        loss.append(energy.item())
        pbar.set_postfix({'Energy': f'{energy.item()}'})

    plt.figure(figsize=(10, 5))
    plt.plot(t_steps, loss, label='Energy variation curve')
    plt.xlabel('Steps')
    plt.ylabel('Ground state Energy')
    plt.title('Energy Variation Curve')
    plt.legend()
    plt.grid()
    plt.show()
    

if __name__ == "__main__":
    calc_TFI_with_MPO()

In [ ]:
import torch as tc
import torch.nn as nn
import numpy as np
import math as mt
from tqdm import tqdm
import matplotlib.pyplot as plt

def choose_device():
    if tc.cuda.is_available():
        return 'cuda'
    else:
        return 'cpu'

def clone_func(tensors):
    """辅助函数：克隆 MPS 张量列表"""
    return [tensor.clone() for tensor in tensors]

class MPS(nn.Module):
    def __init__(self, length, dim, chi, device='cpu', dtype=tc.complex128, boundary='open'):
        super().__init__()
        self.length = length
        self.dim = dim
        self.chi = chi
        self.device = device
        self.dtype = dtype
        self.boundary = boundary
        
        # 将张量注册为模块参数
        if self.boundary == 'open':
            tensors = [tc.randn((1, dim, chi), device=device, dtype=dtype, requires_grad=True)] + [tc.randn((chi, dim, chi), device=device, dtype=dtype, requires_grad=True) for _ in range(length-2)] + [tc.randn((chi, dim, 1), device=device, dtype=dtype, requires_grad=True)]
        elif self.boundary == 'periodic':
            tensors = [tc.randn((chi, dim, chi), device=device, dtype=dtype, requires_grad=True) for _ in range(length)]
        
        # 将张量注册为参数
        self.tensors = nn.ParameterList([nn.Parameter(t, requires_grad=True) for t in tensors])

    def clone_tensors(self):
        return clone_func(self.tensors)

    def clone_mps(self):
        psi = MPS(self.length, self.dim, self.chi, self.device, self.dtype)
        # 复制参数值
        for i, param in enumerate(self.tensors):
            psi.tensors[i].data = param.data.clone()
        return psi

    def normalize(self):
        mps0 = self.clone_tensors()
        v = tc.ones((mps0[0].shape[0], mps0[0].shape[0]), dtype=self.dtype, device=self.device)
        mps_n = []
        norm = tc.zeros(len(self.tensors), dtype=tc.float64, device=self.device)
        
        for i in range(len(self.tensors)):
            v = tc.einsum('ab,acd,bce->de', v, mps0[i].conj(), mps0[i])
            n = tc.norm(v) + 1e-12
            mps_n.append(self.tensors[i] / tc.sqrt(n))
            v = v / n
            norm[i] = n
        
        norms = tc.prod(norm)
        
        # 更新参数值
        for i in range(len(self.tensors)):
            self.tensors[i].data = mps_n[i].data
        
        return norms

    def contract_mps(self):
        """收缩整个 MPS 为单个张量"""
        tensor = self.tensors[0]
        for A in self.tensors[1:]:
            tensor = tc.einsum('...i,ijk->...jk', tensor, A)
        
        if self.boundary == 'open':
            return tensor.squeeze()
        elif self.boundary == 'periodic':
            return tc.einsum('i...i->...', tensor)

    def add_single_opr(self, op, site):
        """在指定位置添加单体操作"""
        return tc.einsum('abc, bd -> adc', self.tensors[site], op)

    def inner_product(self, others, form='log', boundary='open'):
        tensors0 = self.tensors
        tensors1 = others.tensors
        assert tensors0[0].shape[0] == tensors0[-1].shape[-1]
        assert tensors1[0].shape[0] == tensors1[-1].shape[-1]
        assert len(tensors0) == len(tensors1)
        
        if boundary == 'open':
            v = tc.ones((1, 1), dtype=self.dtype, device=self.device)
            norm_tensor = tc.ones(len(tensors0), dtype=self.dtype, device=self.device)
            
            for i in range(len(tensors0)):
                v = tc.einsum('ab,acd,bce->de', v, tensors0[i].conj(), tensors1[i])
                n = tc.norm(v) + 1e-16
                if n.item() < 1e-12:
                    return tc.tensor(0.0, dtype=self.dtype, device=self.device)
                v = v / n
                norm_tensor[i] = n
            
            norms = tc.prod(norm_tensor)
            return norms * v.squeeze()

        elif boundary == 'periodic':
            v0 = tc.eye(tensors0[0].shape[0], dtype=tensors0[0].dtype, device=tensors0[0].device)
            v1 = tc.eye(tensors1[0].shape[0], dtype=tensors0[0].dtype, device=tensors0[0].device)
            v = tc.kron(v0, v1).reshape([tensors0[0].shape[0], tensors1[0].shape[0],
                                        tensors0[0].shape[0], tensors1[0].shape[0]])
            norm_list = list()
            for n in range(len(tensors0)):
                v = tc.einsum('uvap,adb,pdq->uvbq', v, tensors0[n].conj(), tensors1[n])
                norm_list.append(v.norm() + 1e-10)
                v = v / norm_list[-1]
            
            if v.numel() > 1:
                norm1 = tc.einsum('acac->', v)
                norm_list.append(norm1 + 1e-10)
            else:
                norm_list.append(v[0, 0, 0, 0] + 1e-10)
            
            if form == 'log':
                norm = 0.0
                for x in norm_list:
                    norm = norm + tc.log(x.abs())
            elif form == 'list':
                return norm_list
            else:
                norm = 1.0
                for x in norm_list:
                    norm = norm * x
            return norm

    def energy_calcu(self, ops, sites):
        """计算给定操作和位置的能量期望值"""
        assert len(ops) == len(sites), "操作和位置列表长度不匹配"
        energy = 0
        psi = self.clone_mps()
        for op, site in zip(ops, sites):
            psi.tensors[site] = psi.add_single_opr(op, site)
        energy = self.inner_product(psi, boundary=self.boundary)
        return energy
    
    def energy_with_MPO(self, mpo):
        assert self.length == mpo.length, "MPS和MPO长度不匹配"
    
        if self.boundary == 'open':
            v = tc.ones((1, 1, 1), dtype=self.dtype, device=self.device)
            norm_tensor = tc.ones(self.length, dtype=self.dtype, device=self.device)
            
            for i in range(self.length):
                v = tc.einsum('abc, adi, cek, bjde -> ijk', v, 
                             self.tensors[i].conj(), self.tensors[i], mpo.tensors[i])
                n = tc.norm(v) + 1e-16
                if n.item() < 1e-12:
                    return tc.tensor(0.0, dtype=self.dtype, device=self.device)
                v = v / n
                norm_tensor[i] = n
            
            norms = tc.prod(norm_tensor)
            return norms * v.squeeze()
        
        elif self.boundary == 'periodic':
            v0 = tc.eye(self.tensors[0].shape[0], dtype=self.dtype, device=self.device)
            v1 = tc.eye(self.tensors[0].shape[0], dtype=self.dtype, device=self.device)
            v = tc.kron(v0, v1).reshape([self.tensors[0].shape[0], self.tensors[0].shape[0],
                                        self.tensors[0].shape[0], self.tensors[0].shape[0]])
            norm_list = list()
            
            for n in range(self.length):
                v = tc.einsum('uvap,adb,pdq->uvbq', v, self.tensors[n].conj(), self.tensors[n])
                norm_list.append(v.norm() + 1e-10)
                v = v / norm_list[-1]
            
            if v.numel() > 1:
                norm1 = tc.einsum('acac->', v)
                norm_list.append(norm1 + 1e-10)
            else:
                norm_list.append(v[0, 0, 0, 0] + 1e-10)
            
            norm = 1.0
            for x in norm_list:
                norm = norm * x
            return norm


class MPO(nn.Module):
    def __init__(self, length, dim, device='cpu', dtype=tc.complex128, name=None):
        super().__init__()
        self.length = length
        self.dim = dim
        self.device = device
        self.dtype = dtype
        self.name = name
        self.tensors = None
        
    def generate_spin_operators(self):
        """生成自旋维度为dim的所有自旋算符矩阵"""
        S = (self.dim - 1) / 2.0
        m_values = np.arange(S, -S-1, -1)

        Sz = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        S_plus = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        S_minus = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        
        for i, m in enumerate(m_values):
            Sz[i, i] = m
        
        for i, m in enumerate(m_values):
            if i > 0:
                coeff = np.sqrt(S * (S + 1) - m * (m + 1))
                S_plus[i-1, i] = coeff

            if i < self.dim - 1:
                coeff = np.sqrt(S * (S + 1) - m * (m - 1))
                S_minus[i+1, i] = coeff
        
        Sx = (S_plus + S_minus) / 2.0
        Sy = (S_plus - S_minus) / (2.0j)
        I = tc.eye(self.dim, dtype=self.dtype, device=self.device)
        
        return {
            'I': I,
            'Sx': Sx,
            'Sy': Sy, 
            'Sz': Sz,
            'S_plus': S_plus,
            'S_minus': S_minus,
            'S': S,
            'dim': self.dim
        }
        
    def get_spin_operator(self, op_name):
        """获取指定的自旋算符"""
        operators = self.generate_spin_operators()
        
        if op_name in operators:
            self.tensor = operators[op_name]
            self.name = op_name
            return self
        else:
            raise ValueError(f"未知的算符名称: {op_name}")
    
    def get_pauli_matrices(self):
        """获取泡利矩阵 (自旋-1/2)"""
        if self.dim != 2:
            raise ValueError("泡利矩阵只适用于自旋-1/2系统 (dim=2)")
        
        operators = self.generate_spin_operators()
        pauli_x = 2 * operators['Sx']
        pauli_y = 2 * operators['Sy'] 
        pauli_z = 2 * operators['Sz']
        
        return {
            'I': operators['I'],
            'sigma_x': pauli_x,
            'sigma_y': pauli_y,
            'sigma_z': pauli_z,
            'sigma_plus': pauli_x + 1j * pauli_y,
            'sigma_minus': pauli_x - 1j * pauli_y
        }
    
    def generate_TFI_MPO(self, J, h):
        """生成横场Ising模型的MPO张量列表"""
        self.tensors = [None] * self.length
        d = self.dim
        D = 3  # 虚拟维度
        
        for i in range(self.length):
            mpo_tensor = tc.zeros((D, D, d, d), dtype=self.dtype, device=self.device)
            
            # 单位算符部分
            mpo_tensor[0, 0, :, :] = self.get_spin_operator('I')
            mpo_tensor[2:, 2:, :, :] = self.get_spin_operator('I')

            # Sz算符
            mpo_tensor[0, 1, :, :] = mt.sqrt(J) * self.get_spin_operator('Sz')
            mpo_tensor[1, 2, :, :] = - mt.sqrt(J) * self.get_spin_operator('Sz')

            # Sx算符
            mpo_tensor[0, 2, :, :] = h * self.get_spin_operator('Sx')
            
            # 边界条件处理
            if i == 0:
                mpo_tensor = mpo_tensor[0:1, :, :, :]
            if i == self.length - 1:
                mpo_tensor = mpo_tensor[:, 2:3, :, :]
            
            self.tensors[i] = mpo_tensor
            
        return self


class TFIModel(nn.Module):
    """横场Ising模型的变分基态求解器"""
    def __init__(self, length, dim, chi, J, h, device='cpu', dtype=tc.float64):
        super().__init__()
        self.length = length
        self.dim = dim
        self.chi = chi
        self.J = J
        self.h = h
        self.device = device
        self.dtype = dtype
        
        # 初始化MPS和MPO
        self.mps = MPS(length, dim, chi, device, dtype)
        self.mpo = MPO(length, dim, device, dtype)
        self.mpo.generate_TFI_MPO(J, h)
        
    def forward(self):
        """前向传播：计算能量期望值"""
        return self.mps.energy_with_MPO(self.mpo)
    
    def normalize(self):
        """归一化MPS"""
        return self.mps.normalize()
    
    def optimize(self, steps=10000, lr=1e-4):
        """优化MPS以找到基态"""
        optimizer = tc.optim.Adam(self.mps.parameters(), lr=lr)
        energies = []
        
        pbar = tqdm(range(steps), desc="Variational DMRG", unit="step")
        
        for step in pbar:
            self.normalize()
            energy = self.forward()
            
            optimizer.zero_grad()
            energy.backward()
            optimizer.step()
            
            energies.append(energy.item())
            pbar.set_postfix({'Energy': f'{energy.item()}'})
        
        return energies
    
    def plot_energy_curve(self, energies):
        """绘制能量变化曲线"""
        plt.figure(figsize=(10, 5))
        plt.plot(range(1, len(energies)+1), energies, label='Energy variation curve')
        plt.xlabel('Steps')
        plt.ylabel('Ground state Energy')
        plt.title('Energy Variation Curve')
        plt.legend()
        plt.grid()
        plt.show()


def calc_TFI_with_MPO():
    """使用TFIModel类计算横场Ising模型的基态能量"""
    model = TFIModel(length=20, dim=2, chi=50, J=1.0, h=0.5, 
                    device=choose_device(), dtype=tc.float64)
    energies = model.optimize(steps=10000)
    model.plot_energy_curve(energies)
    

if __name__ == "__main__":
    calc_TFI_with_MPO()

In [ ]:
import torch as tc
import torch.nn as nn
import numpy as np
import math as mt
from tqdm import tqdm
import matplotlib.pyplot as plt

def choose_device():
    if tc.cuda.is_available():
        return 'cuda'
    else:
        return 'cpu'

class MPS(nn.Module):
    def __init__(self, length, dim, chi, device=choose_device(), dtype=tc.complex128, boundary='open'):
        super().__init__()
        self.length = length
        self.dim = dim
        self.chi = chi
        self.device = device
        self.dtype = dtype
        self.boundary = boundary
        
        # 将张量注册为模块参数
        if self.boundary == 'open':
            tensors = [tc.randn((1, dim, chi), device=device, dtype=dtype, requires_grad=True)] + \
                     [tc.randn((chi, dim, chi), device=device, dtype=dtype, requires_grad=True) for _ in range(length-2)] + \
                     [tc.randn((chi, dim, 1), device=device, dtype=dtype, requires_grad=True)]
        elif self.boundary == 'periodic':
            tensors = [tc.randn((chi, dim, chi), device=device, dtype=dtype, requires_grad=True) for _ in range(length)]
        
        # 将张量注册为参数
        self.tensors = nn.ParameterList([nn.Parameter(t, requires_grad=True) for t in tensors])

    def get_normalized_tensors(self):
        """
        获取归一化的张量，保持计算图
        使用可微分的归一化方法
        """
        normalized_tensors = []
        norm_accumulator = tc.tensor(1.0, dtype=tc.float64, device=self.device, requires_grad=True)
        
        # 计算总的归一化因子（保持梯度）
        total_norm_squared = tc.tensor(0.0, dtype=tc.float64, device=self.device, requires_grad=True)
        
        # 首先计算所有张量的范数平方和
        for tensor in self.tensors:
            total_norm_squared = total_norm_squared + tc.sum(tc.abs(tensor)**2)
        
        # 计算归一化因子
        norm_factor = tc.sqrt(total_norm_squared / self.length) + 1e-12
        
        # 归一化每个张量
        for tensor in self.tensors:
            normalized_tensors.append(tensor / norm_factor)
            
        return normalized_tensors

    def get_canonical_form(self, center=None):
        """
        获取标准形式的MPS，保持计算图
        这是一个更稳定的归一化方法
        """
        if center is None:
            center = self.length // 2
            
        normalized_tensors = list(self.tensors)  # 保持原始引用
        
        # 从左到右正交化到center
        for i in range(center):
            tensor = normalized_tensors[i]
            # 重塑为矩阵
            if i == 0:
                shape = tensor.shape
                matrix = tensor.view(shape[1], -1)  # (d, chi)
            else:
                shape = tensor.shape
                matrix = tensor.view(-1, shape[-1])  # (chi_l * d, chi_r)
            
            # QR分解（可微分）
            Q, R = tc.linalg.qr(matrix)
            
            # 更新当前张量
            if i == 0:
                normalized_tensors[i] = Q.view(1, shape[1], -1)
            else:
                normalized_tensors[i] = Q.view(shape[0], shape[1], -1)
            
            # 将R传递给下一个张量
            if i < center:
                next_shape = normalized_tensors[i+1].shape
                if i == center - 1:
                    # 最后一个，乘到center张量上
                    normalized_tensors[i+1] = tc.einsum('ij,jkl->ikl', R, normalized_tensors[i+1])
                else:
                    # 乘到下一个张量
                    normalized_tensors[i+1] = tc.einsum('ij,jkl->ikl', R, normalized_tensors[i+1])
        
        return normalized_tensors

    def compute_overlap_matrix(self, normalized_tensors):
        """计算重叠矩阵，用于归一化检查"""
        if self.boundary == 'open':
            v = tc.ones((1, 1), dtype=self.dtype, device=self.device)
            
            for i in range(self.length):
                tensor = normalized_tensors[i]
                v = tc.einsum('ab,acd,bce->de', v, tensor.conj(), tensor)
            
            return v.squeeze()
        
        elif self.boundary == 'periodic':
            v = tc.eye(normalized_tensors[0].shape[0], dtype=self.dtype, device=self.device)
            
            for tensor in normalized_tensors:
                # 对周期边界的处理
                v = tc.einsum('ab,acd,bce->de', v, tensor.conj(), tensor)
            
            return tc.trace(v)

    def energy_with_MPO_normalized(self, mpo, use_canonical=True):
        """
        计算归一化MPS与MPO的能量期望值，保持梯度
        """
        if use_canonical:
            normalized_tensors = self.get_canonical_form()
        else:
            normalized_tensors = self.get_normalized_tensors()
        
        if self.boundary == 'open':
            # 计算 <psi|H|psi>
            v_H = tc.ones((1, 1, 1), dtype=self.dtype, device=self.device)
            
            for i in range(self.length):
                # 三维收缩：bra, MPO, ket
                v_H = tc.einsum('abc,adi,cek,bjde->ijk', 
                               v_H, 
                               normalized_tensors[i].conj(), 
                               normalized_tensors[i], 
                               mpo.tensors[i])
                
                # 稳定化（保持梯度）
                norm = tc.norm(v_H) + 1e-16
                v_H = v_H / norm
            
            energy = v_H.squeeze()
            
            # 计算 <psi|psi> 用于归一化检查
            overlap = self.compute_overlap_matrix(normalized_tensors)
            
            # 返回归一化的能量
            return energy / (overlap + 1e-16)
        
        else:  # periodic boundary
            # 类似实现...
            raise NotImplementedError("Periodic boundary not implemented for normalized energy")

    def contract_normalized_mps(self, use_canonical=True):
        """收缩归一化的MPS为单个张量"""
        if use_canonical:
            normalized_tensors = self.get_canonical_form()
        else:
            normalized_tensors = self.get_normalized_tensors()
            
        tensor = normalized_tensors[0]
        for A in normalized_tensors[1:]:
            tensor = tc.einsum('...i,ijk->...jk', tensor, A)
        
        if self.boundary == 'open':
            return tensor.squeeze()
        elif self.boundary == 'periodic':
            return tc.einsum('i...i->...', tensor)


class MPO(nn.Module):
    def __init__(self, length, dim, device=choose_device(), dtype=tc.complex128, name=None):
        super().__init__()
        self.length = length
        self.dim = dim
        self.device = device
        self.dtype = dtype
        self.name = name
        self.tensors = None
        
    def generate_spin_operators(self):
        """生成自旋维度为dim的所有自旋算符矩阵"""
        S = (self.dim - 1) / 2.0
        m_values = np.arange(S, -S-1, -1)

        Sz = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        S_plus = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        S_minus = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        
        for i, m in enumerate(m_values):
            Sz[i, i] = m
        
        for i, m in enumerate(m_values):
            if i > 0:
                coeff = np.sqrt(S * (S + 1) - m * (m + 1))
                S_plus[i-1, i] = coeff

            if i < self.dim - 1:
                coeff = np.sqrt(S * (S + 1) - m * (m - 1))
                S_minus[i+1, i] = coeff
        
        Sx = (S_plus + S_minus) / 2.0
        Sy = (S_plus - S_minus) / (2.0j)
        I = tc.eye(self.dim, dtype=self.dtype, device=self.device)
        
        return {
            'I': I, 'Sx': Sx, 'Sy': Sy, 'Sz': Sz,
            'S_plus': S_plus, 'S_minus': S_minus,
            'S': S, 'dim': self.dim
        }
        
    def get_spin_operator(self, op_name):
        """获取指定的自旋算符"""
        operators = self.generate_spin_operators()
        if op_name in operators:
            return operators[op_name]
        else:
            raise ValueError(f"未知的算符名称: {op_name}")
    
    def generate_TFI_MPO(self, J, h):
        """生成横场Ising模型的MPO张量列表"""
        self.tensors = [None] * self.length
        d = self.dim
        D = 3  # 虚拟维度
        
        for i in range(self.length):
            mpo_tensor = tc.zeros((D, D, d, d), dtype=self.dtype, device=self.device)
            
            # 单位算符部分
            mpo_tensor[0, 0, :, :] = self.get_spin_operator('I')
            mpo_tensor[2, 2, :, :] = self.get_spin_operator('I')

            # Sz算符
            mpo_tensor[0, 1, :, :] = mt.sqrt(J) * self.get_spin_operator('Sz')
            mpo_tensor[1, 2, :, :] = -mt.sqrt(J) * self.get_spin_operator('Sz')

            # Sx算符
            mpo_tensor[0, 2, :, :] = h * self.get_spin_operator('Sx')
            
            # 边界条件处理
            if i == 0:
                mpo_tensor = mpo_tensor[0:1, :, :, :]
            if i == self.length - 1:
                mpo_tensor = mpo_tensor[:, 2:3, :, :]
            
            self.tensors[i] = mpo_tensor
            
        return self


class ImprovedDMRGnet(nn.Module):
    """改进的DMRG收缩网络，支持稳定的归一化"""
    def __init__(self, mps, mpo, normalization_method='canonical'):
        super().__init__()
        self.mps = mps
        self.mpo = mpo
        self.normalization_method = normalization_method  # 'canonical' or 'simple'
        
    def forward(self):
        """前向传播：计算归一化MPS的能量期望值"""
        use_canonical = (self.normalization_method == 'canonical')
        return self.mps.energy_with_MPO_normalized(self.mpo, use_canonical=use_canonical)
    
    def get_normalized_mps_state(self):
        """获取当前的归一化MPS状态"""
        use_canonical = (self.normalization_method == 'canonical')
        return self.mps.contract_normalized_mps(use_canonical=use_canonical)
    
    def optimize(self, steps=10000, lr=1e-4, scheduler_step=1000):
        """优化MPS以找到基态，添加学习率调度"""
        optimizer = tc.optim.Adam(self.mps.parameters(), lr=lr)
        scheduler = tc.optim.lr_scheduler.StepLR(optimizer, step_size=scheduler_step, gamma=0.9)
        
        energies = []
        
        pbar = tqdm(range(steps), desc="Improved Variational DMRG", unit="step")
        
        for step in pbar:
            energy = self.forward()
            
            optimizer.zero_grad()
            # 检查能量是否为实数（对于厄米哈密顿量）
            if tc.is_complex(energy):
                energy_real = energy.real
            else:
                energy_real = energy
                
            energy_real.backward()
            
            # 梯度裁剪，防止梯度爆炸
            tc.nn.utils.clip_grad_norm_(self.mps.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            energies.append(energy_real.item())
            
            # 每1000步检查一次归一化
            if step % 1000 == 0:
                with tc.no_grad():
                    norm = self.mps.compute_overlap_matrix(
                        self.mps.get_canonical_form() if self.normalization_method == 'canonical' 
                        else self.mps.get_normalized_tensors()
                    )
                    pbar.set_postfix({
                        'Energy': f'{energy_real.item():.6f}', 
                        'Norm': f'{norm.real.item():.6f}',
                        'LR': f'{scheduler.get_last_lr()[0]:.2e}'
                    })
            else:
                pbar.set_postfix({'Energy': f'{energy_real.item():.6f}'})
        
        return energies
    
    def plot_energy_curve(self, energies, title="Energy Variation Curve"):
        """绘制能量变化曲线"""
        plt.figure(figsize=(12, 8))
        
        # 主图
        plt.subplot(2, 1, 1)
        plt.plot(range(1, len(energies)+1), energies, 'b-', linewidth=1, alpha=0.7)
        plt.xlabel('Steps')
        plt.ylabel('Ground State Energy')
        plt.title(f'{title} - Full Range')
        plt.grid(True, alpha=0.3)
        
        # 收敛部分放大图
        plt.subplot(2, 1, 2)
        if len(energies) > 1000:
            convergence_start = len(energies) // 2
            conv_energies = energies[convergence_start:]
            plt.plot(range(convergence_start, len(energies)), conv_energies, 'r-', linewidth=1)
            plt.xlabel('Steps')
            plt.ylabel('Ground State Energy')
            plt.title('Convergence Region (Second Half)')
            plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()


def improved_AD_DMRG(N, d, chi, J=1.0, h=0.5, device=choose_device(), dtype=tc.complex128, 
                     normalization_method='canonical', steps=10000, lr=1e-4):
    """
    改进的自动微分DMRG实现
    
    参数:
    - normalization_method: 'canonical' 或 'simple'
    - 其他参数同原版本
    """
    print(f"初始化系统: N={N}, d={d}, chi={chi}, J={J}, h={h}")
    print(f"归一化方法: {normalization_method}")
    
    # 初始化MPS和MPO
    mps = MPS(length=N, dim=d, chi=chi, device=device, dtype=dtype)
    mpo = MPO(length=N, dim=d, device=device, dtype=dtype)
    mpo.generate_TFI_MPO(J, h)
    
    # 创建改进的DMRG网络
    model = ImprovedDMRGnet(mps, mpo, normalization_method=normalization_method)
    
    # 优化
    energies = model.optimize(steps=steps, lr=lr)
    
    # 绘制结果
    model.plot_energy_curve(energies, title=f"Improved DMRG ({normalization_method} normalization)")
    
    print(f"最终基态能量: {energies[-1]:.8f}")
    return model, energies

# 使用示例
if __name__ == "__main__":
    # 测试改进版本
    model, energies = improved_AD_DMRG(N=10, d=2, chi=16, J=1.0, h=0.5, 
                                      normalization_method='canonical', 
                                      steps=5000, lr=1e-3)

In [ ]:
import torch as tc
import torch.nn as nn
import numpy as np
import math as mt
from tqdm import tqdm
import matplotlib.pyplot as plt

def choose_device():
    if tc.cuda.is_available():
        return 'cuda'
    else:
        return 'cpu'

def clone_func(tensors):
    """辅助函数：克隆 MPS 张量列表"""
    return [tensor.clone() for tensor in tensors]

def check_paras(mps, mpo):
    assert mps.length == mpo.length, "MPS和MPO长度不匹配"
    assert mps.dim == mpo.dim, "MPS和MPO维度不匹配"
    assert mps.dtype == mpo.dtype, "MPS和MPO数据类型不匹配"
    assert mps.device == mpo.device, "MPS和MPO设备不匹配"
    assert mpo.tensors is not None, "MPO 尚未生成张量（请先调用 generate_TFI_MPO）"

class MPS(nn.Module):
    def __init__(self, length, dim, chi, device=choose_device(), dtype=tc.complex128, boundary='open'):
        super().__init__()
        self.length = length
        self.dim = dim
        self.chi = chi
        self.device = device
        self.dtype = dtype
        self.boundary = boundary
        
        # 将张量注册为模块参数
        if self.boundary == 'open':
            tensors = [tc.randn((1, dim, chi), device=device, dtype=dtype, requires_grad=True)] + [tc.randn((chi, dim, chi), device=device, dtype=dtype, requires_grad=True) for _ in range(length-2)] + [tc.randn((chi, dim, 1), device=device, dtype=dtype, requires_grad=True)]
        elif self.boundary == 'periodic':
            tensors = [tc.randn((chi, dim, chi), device=device, dtype=dtype, requires_grad=True) for _ in range(length)]
        
        # 将张量注册为参数
        self.tensors = nn.ParameterList([nn.Parameter(t, requires_grad=True) for t in tensors])

    def clone_tensors(self):
        return clone_func(self.tensors)

    def clone_mps(self):
        psi = MPS(self.length, self.dim, self.chi, self.device, self.dtype)
        # 复制参数值
        for i, param in enumerate(self.tensors):
            psi.tensors[i].data = param.data.clone()
        return psi

    def normalize(self):
        normalize_tensors = []
        mps0 = self.clone_tensors()
        v = tc.ones((mps0[0].shape[0], mps0[0].shape[0]), dtype=self.dtype, device=self.device)
        mps_n = []
        norm = tc.zeros(len(self.tensors), dtype=tc.float64, device=self.device)
        
        for i in range(len(self.tensors)):
            v = tc.einsum('ab,acd,bce->de', v, mps0[i].conj(), mps0[i])
            n = tc.norm(v) + 1e-12
            normalize_tensors.append(mps0[i] / n)
            v = v / n
            norm[i] = n

        return normalize_tensors

    def contract_mps(self):
        """收缩整个 MPS 为单个张量"""
        tensor = self.tensors[0]
        for A in self.tensors[1:]:
            tensor = tc.einsum('...i,ijk->...jk', tensor, A)
        
        if self.boundary == 'open':
            return tensor.squeeze()
        elif self.boundary == 'periodic':
            return tc.einsum('i...i->...', tensor)

    def add_single_opr(self, op, site):
        """在指定位置添加单体操作"""
        return tc.einsum('abc, bd -> adc', self.tensors[site], op)

    def inner_product(self, others, form='log', boundary='open'):
        tensors0 = self.tensors
        tensors1 = others.tensors
        assert tensors0[0].shape[0] == tensors0[-1].shape[-1]
        assert tensors1[0].shape[0] == tensors1[-1].shape[-1]
        assert len(tensors0) == len(tensors1)
        
        if boundary == 'open':
            v = tc.ones((1, 1), dtype=self.dtype, device=self.device)
            norm_tensor = tc.ones(len(tensors0), dtype=self.dtype, device=self.device)
            
            for i in range(len(tensors0)):
                v = tc.einsum('ab,acd,bce->de', v, tensors0[i].conj(), tensors1[i])
                n = tc.norm(v) + 1e-16
                if n.item() < 1e-12:
                    return tc.tensor(0.0, dtype=self.dtype, device=self.device)
                v = v / n
                norm_tensor[i] = n
            
            norms = tc.prod(norm_tensor)
            return norms * v.squeeze()

        elif boundary == 'periodic':
            v0 = tc.eye(tensors0[0].shape[0], dtype=tensors0[0].dtype, device=tensors0[0].device)
            v1 = tc.eye(tensors1[0].shape[0], dtype=tensors0[0].dtype, device=tensors0[0].device)
            v = tc.kron(v0, v1).reshape([tensors0[0].shape[0], tensors1[0].shape[0],
                                        tensors0[0].shape[0], tensors1[0].shape[0]])
            norm_list = list()
            for n in range(len(tensors0)):
                v = tc.einsum('uvap,adb,pdq->uvbq', v, tensors0[n].conj(), tensors1[n])
                norm_list.append(v.norm() + 1e-10)
                v = v / norm_list[-1]
            
            if v.numel() > 1:
                norm1 = tc.einsum('acac->', v)
                norm_list.append(norm1 + 1e-10)
            else:
                norm_list.append(v[0, 0, 0, 0] + 1e-10)
            
            if form == 'log':
                norm = 0.0
                for x in norm_list:
                    norm = norm + tc.log(x.abs())
            elif form == 'list':
                return norm_list
            else:
                norm = 1.0
                for x in norm_list:
                    norm = norm * x
            return norm
    
    def energy_with_MPO(self, mpo):
        assert self.length == mpo.length, "MPS和MPO长度不匹配"
    
        if self.boundary == 'open':
            v = tc.ones((1, 1, 1), dtype=self.dtype, device=self.device)
            norm_tensor = tc.ones(self.length, dtype=self.dtype, device=self.device)

            norm_tensors = self.normalize()
            
            for i in range(self.length):
                v = tc.einsum('abc, adi, cek, bjde -> ijk', v, 
                             norm_tensors[i].conj(), norm_tensors[i], mpo.tensors[i])
                n = tc.norm(v) + 1e-16
                if n.item() < 1e-12:
                    return tc.tensor(0.0, dtype=self.dtype, device=self.device)
                v = v / n
                norm_tensor[i] = n
            
            norms = tc.prod(norm_tensor)
            return norms * v.squeeze(), norm_tensors
        
        elif self.boundary == 'periodic':
            v0 = tc.eye(self.tensors[0].shape[0], dtype=self.dtype, device=self.device)
            v1 = tc.eye(self.tensors[0].shape[0], dtype=self.dtype, device=self.device)
            v = tc.kron(v0, v1).reshape([self.tensors[0].shape[0], self.tensors[0].shape[0],
                                        self.tensors[0].shape[0], self.tensors[0].shape[0]])
            norm_list = list()
            
            for n in range(self.length):
                v = tc.einsum('uvap,adb,pdq->uvbq', v, self.tensors[n].conj(), self.tensors[n])
                norm_list.append(v.norm() + 1e-10)
                v = v / norm_list[-1]
            
            if v.numel() > 1:
                norm1 = tc.einsum('acac->', v)
                norm_list.append(norm1 + 1e-10)
            else:
                norm_list.append(v[0, 0, 0, 0] + 1e-10)
            
            norm = 1.0
            for x in norm_list:
                norm = norm * x
            return norm
        
    def to_normalize(self):
        "用于训练后的结果归一化"
        psi = self.clone_mps()
        with tc.no_grad():
            normalized_tensors = self.normalize()
            for i in range(self.length):
                psi.tensors[i].data = normalized_tensors[i]
        return psi
        
    def measurement_calcu(self, ops, sites):
        "用于计算训练后测量值"
        assert self.dim == ops[0].shape[0]
        assert len(ops) == len(sites), "操作和位置列表长度不匹配"

        measurement = 0
        with tc.no_grad():
            psi = self.to_normalize()
            psi_ = psi.clone_mps()
            for op, site in zip(ops, sites):
                psi_.tensors[site] = psi_.add_single_opr(op, site)
            measurement = psi.inner_product(psi_, boundary=self.boundary)
        return measurement
                 


class MPO(nn.Module):
    def __init__(self, length, dim, device=choose_device(), dtype=tc.complex128, name=None):
        super().__init__()
        self.length = length
        self.dim = dim
        self.device = device
        self.dtype = dtype
        self.name = name
        self.tensors = None
        
    def generate_spin_operators(self):
        """生成自旋维度为dim的所有自旋算符矩阵"""
        S = (self.dim - 1) / 2.0
        m_values = np.arange(S, -S-1, -1)

        Sz = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        S_plus = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        S_minus = tc.zeros((self.dim, self.dim), dtype=self.dtype, device=self.device)
        
        for i, m in enumerate(m_values):
            Sz[i, i] = m
        
        for i, m in enumerate(m_values):
            if i > 0:
                coeff = np.sqrt(S * (S + 1) - m * (m + 1))
                S_plus[i-1, i] = coeff

            if i < self.dim - 1:
                coeff = np.sqrt(S * (S + 1) - m * (m - 1))
                S_minus[i+1, i] = coeff
        
        Sx = (S_plus + S_minus) / 2.0
        Sy = (S_plus - S_minus) / (2.0j)
        I = tc.eye(self.dim, dtype=self.dtype, device=self.device)
        
        return {
            'I': I,
            'Sx': Sx,
            'Sy': Sy, 
            'Sz': Sz,
            'S_plus': S_plus,
            'S_minus': S_minus,
            'S': S,
            'dim': self.dim
        }
        
    def get_spin_operator(self, op_name):
        """获取指定的自旋算符"""
        operators = self.generate_spin_operators()
        
        if op_name in operators:
            return operators[op_name]
        else:
            raise ValueError(f"未知的算符名称: {op_name}")
    
    def get_pauli_matrices(self):
        """获取泡利矩阵 (自旋-1/2)"""
        if self.dim != 2:
            raise ValueError("泡利矩阵只适用于自旋-1/2系统 (dim=2)")
        
        operators = self.generate_spin_operators()
        pauli_x = 2 * operators['Sx']
        pauli_y = 2 * operators['Sy'] 
        pauli_z = 2 * operators['Sz']
        
        return {
            'I': operators['I'],
            'sigma_x': pauli_x,
            'sigma_y': pauli_y,
            'sigma_z': pauli_z,
            'sigma_plus': pauli_x + 1j * pauli_y,
            'sigma_minus': pauli_x - 1j * pauli_y
        }
    
    def generate_TFI_MPO(self, J, h):
        """生成横场Ising模型的MPO张量列表"""
        self.tensors = [None] * self.length
        d = self.dim
        D = 3  # 虚拟维度
        
        for i in range(self.length):
            mpo_tensor = tc.zeros((D, D, d, d), dtype=self.dtype, device=self.device)
            
            # 单位算符部分
            mpo_tensor[0, 0, :, :] = self.get_spin_operator('I')
            mpo_tensor[2:, 2:, :, :] = self.get_spin_operator('I')

            # Sz算符
            mpo_tensor[0, 1, :, :] = mt.sqrt(J) * self.get_spin_operator('Sz')
            mpo_tensor[1, 2, :, :] = - mt.sqrt(J) * self.get_spin_operator('Sz')

            # Sx算符
            mpo_tensor[0, 2, :, :] = h * self.get_spin_operator('Sx')
            
            # 边界条件处理
            if i == 0:
                mpo_tensor = mpo_tensor[0:1, :, :, :]
            if i == self.length - 1:
                mpo_tensor = mpo_tensor[:, 2:3, :, :]
            
            self.tensors[i] = mpo_tensor
            
        return self
    

#任意的AD——DMRG方法类
class DMRGnet(nn.Module):
    """任意的DMRG收缩网络"""
    def __init__(self, mps, mpo):
        super().__init__()
        check_paras(mps, mpo)
        self.length = mps.length
        self.dim = mps.dim
        self.chi = mps.chi
        self.device = mps.device
        self.dtype = mps.dtype
        
        # 初始化MPS和MPO
        self.mps = mps
        self.mpo = mpo
        
    def forward(self):
        """前向传播：计算能量期望值"""
        return self.energy_with_MPO(self.mpo)
    
    
    def optimize(self, steps=10000, lr=1e-4):
        """优化MPS以找到基态"""
        optimizer = tc.optim.Adam(self.mps.parameters(), lr=lr)
        energies = []
        
        pbar = tqdm(range(steps), desc="Variational DMRG", unit="step")
        
        for step in pbar:
            energy, _ = self.forward()
            
            optimizer.zero_grad()
            energy.backward()
            optimizer.step()
            
            energies.append(energy.item())
            pbar.set_postfix({'Energy': f'{energy.item()}'})
        
        return energies
    
    def plot_energy_curve(self, energies):
        """绘制能量变化曲线"""
        plt.figure(figsize=(10, 5))
        plt.plot(range(1, len(energies)+1), energies, label='Energy variation curve')
        plt.xlabel('Steps')
        plt.ylabel('Ground state Energy')
        plt.title('Energy Variation Curve')
        plt.legend()
        plt.grid()
        plt.show()

def AD_DMRG(N, d, chi, device = choose_device(), dtype=tc.complex128):
    mps = MPS(length=N, dim=d, chi=chi, device=device, dtype=dtype)
    mpo = MPO(length=N, dim=d, device=device, dtype=dtype)
    
    model = DMRGnet(mps, mpo)
    energy = model.optimize(steps=10000, lr=1e-4)
    model.plot_energy_curve(energy)

In [ ]:
import AD_MPS as admps
import torch as tc
import numpy as np
import matplotlib.pyplot as plt

mps = admps.MPS(length=2, dim=2, chi=6, device=admps.choose_device(), dtype=tc.float64)
mpo = admps.MPO(length=2, dim=2, device=admps.choose_device(), dtype=tc.float64).generate_TFI_MPO(J=1.0, h=1.0)

admps.AD_DMRG(mps=mps, mpo=mpo, N=2, d=2, chi=6, device=admps.choose_device(), dtype=tc.float64)

In [ ]:
import numpy as np

# 泡利矩阵
sigma_z = np.array([[1, 0], [0, -1]])
sigma_x = np.array([[0, 1], [1, 0]])
I = np.eye(2)

# 构造哈密顿量
H_zz = - (np.kron(np.kron(sigma_z, sigma_z), I) + np.kron(I, np.kron(sigma_z, sigma_z)))
H_x = - (np.kron(np.kron(sigma_x, I), I) + np.kron(np.kron(I, sigma_x), I) + np.kron(np.kron(I, I), sigma_x))
H = H_zz + H_x

# 对角化
eigenvalues = np.linalg.eigvalsh(H)
print("本征值 (J=1, h=1):")
for i, eig in enumerate(eigenvalues):
    print(f"E{i+1} = {eig:.6f}")

print(f"\n基态能量: {eigenvalues[0]:.6f}")
print(f"第一激发态能量: {eigenvalues[1]:.6f}")
print(f"能隙: {eigenvalues[1] - eigenvalues[0]:.6f}")

现在看来这个能量计算显然存在问题，我们需要构造一个测试态

In [14]:
import AD_MPS as admps
import torch as tc
import numpy as np

J = 1.0
h = 1.0

#构造自旋全为up的直积态：
tensor0 = [tc.zeros((1,2,1), dtype = tc.float64) for _ in range(3)]

for i in range(len(tensor0)):
    tensor0[i][0,0,0]=1.0

t = tc.einsum('abc,cde,efg -> abdfg', tensor0[0], tensor0[1], tensor0[2]).reshape(2,2,2)
print(tc.norm(t))

mps_0 = admps.MPS(tensors=tensor0, length=3, dim=2, chi=1, device=admps.choose_device(), dtype=tc.float64)

norm_tensors = mps_0.normalize()

nt = tc.einsum('abc,cde,efg -> abdfg', norm_tensors[0], norm_tensors[1], norm_tensors[2]).reshape(2,2,2)
print(tc.norm(nt))

'''mpo = admps.MPO(length=3, dim=2, device=admps.choose_device(), dtype=tc.float64).generate_TFI_MPO(J=1.0, h=1.0)

operators = [tc.zeros((1,3,2,2),dtype = tc.float64)] + [tc.zeros((3,3,2,2), dtype = tc.float64)] + [tc.zeros((3,1,2,2), dtype = tc.float64)]

I = tc.eye(mps_0.dim, dtype = tc.float64)
Z = tc.tensor([[1.0, 0.0], [0.0, -1.0]], dtype=tc.float64)
X = tc.tensor([[0.0, 1.0], [1.0, 0.0]], dtype=tc.float64)

operators[0][:,0,:,:] = I
operators[0][:,1,:,:] = Z
operators[0][:,2,:,:] = -h * X

operators[1][0:0,:,:] = I
operators[1][0:1,:,:] = Z
operators[1][0,2,:,:] = -h * X
operators[1][1,2,:,:] = -J * Z
operators[1][2,2,:,:] = I

operators[2][0,:,:,:] = -h * X
operators[2][1,:,:,:] = -J * Z
operators[2][2,:,:,:] = I

e_sum = tc.einsum('abc,cde,efg,ijk,klm,mno,pqbj,qrdl,rsfn -> apigso', tensor0[0].conj(), tensor0[1].conj(), tensor0[2].conj(),tensor0[0], tensor0[1], tensor0[2], operators[0], operators[1], operators[2])

print(e_sum.item())

energy, _ = mps_0.energy_with_MPO(mpo)

print(energy.item())'''


tensor(1., dtype=torch.float64)
tensor(1., dtype=torch.float64, grad_fn=<LinalgVectorNormBackward0>)


"mpo = admps.MPO(length=3, dim=2, device=admps.choose_device(), dtype=tc.float64).generate_TFI_MPO(J=1.0, h=1.0)\n\noperators = [tc.zeros((1,3,2,2),dtype = tc.float64)] + [tc.zeros((3,3,2,2), dtype = tc.float64)] + [tc.zeros((3,1,2,2), dtype = tc.float64)]\n\nI = tc.eye(mps_0.dim, dtype = tc.float64)\nZ = tc.tensor([[1.0, 0.0], [0.0, -1.0]], dtype=tc.float64)\nX = tc.tensor([[0.0, 1.0], [1.0, 0.0]], dtype=tc.float64)\n\noperators[0][:,0,:,:] = I\noperators[0][:,1,:,:] = Z\noperators[0][:,2,:,:] = -h * X\n\noperators[1][0:0,:,:] = I\noperators[1][0:1,:,:] = Z\noperators[1][0,2,:,:] = -h * X\noperators[1][1,2,:,:] = -J * Z\noperators[1][2,2,:,:] = I\n\noperators[2][0,:,:,:] = -h * X\noperators[2][1,:,:,:] = -J * Z\noperators[2][2,:,:,:] = I\n\ne_sum = tc.einsum('abc,cde,efg,ijk,klm,mno,pqbj,qrdl,rsfn -> apigso', tensor0[0].conj(), tensor0[1].conj(), tensor0[2].conj(),tensor0[0], tensor0[1], tensor0[2], operators[0], operators[1], operators[2])\n\nprint(e_sum.item())\n\nenergy, _ = mps_0.